In [16]:
# Reload modules automatically
%load_ext autoreload
%autoreload 2

In [1]:
import cma
import numpy as np
from numbers import Real
import torch
from PIL import Image
import random
import requests
from io import BytesIO
import time
from matplotlib import cm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from tqdm import tqdm
from os import path
import os
import json
import numpy as np
import itertools
import ipywidgets as widgets
from IPython.display import display, clear_output


# Utils imports
from utils.rasterize import *
from utils.load import get_segment_imgs, get_primitive_imgs
from utils.clipemb import CLIP_emb_from_IMG, CLIP_emb_from_TEXT
from utils.cma import clip_sol


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

WIDTH = 300
HEIGHT = 300

CONTENT_DIR = './data'
RESULTS_DIR = path.join(CONTENT_DIR, 'results')
SEGMENT_DIR = path.join(CONTENT_DIR, 'segment_img')
RW_PRIMITIVES_DIR = path.join(CONTENT_DIR, 'rw_primitives')
PRIMITIVE_DIR = path.join(CONTENT_DIR, 'primitives')

SEGMENT_IMGS = get_segment_imgs(SEGMENT_DIR, (WIDTH, HEIGHT))
RW_PRIMITIVES_IMGS = get_segment_imgs(RW_PRIMITIVES_DIR, (WIDTH, HEIGHT))
PRIMITIVE_IMGS = get_primitive_imgs(CONTENT_DIR, (WIDTH, HEIGHT))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.50, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
prompt_text = 'a person is standing under a rainbow colored umbrella covering his head'

primitives_selected = list(RW_PRIMITIVES_IMGS[prompt_text].values())
prompt_embedding = CLIP_emb_from_TEXT(prompt_text)

In [3]:
def compute_loss(prompt_embedding, rasterized_imgs):
        print("------- LOSS -------")
        print("Stacking...")
        rasterized_imgs = np.stack(rasterized_imgs, axis=0) # (population_size, 300, 300, 4)
        print("Permuting to device...")
        rasterized_imgs = torch.tensor(rasterized_imgs).permute(0, 3, 1, 2).to(DEVICE)
        print("Embedding...")
        rasterized_emb = CLIP_emb_from_IMG(rasterized_imgs) # (population_size, 512)
        print("Computing the similarity...")
        similarity = torch.nn.functional.cosine_similarity(rasterized_emb, prompt_embedding, dim=1)
        print("------- DONE -------")
        return similarity

In [4]:
n = 6
min_val = 0.3
max_val = 0.7

# Create a 1D array for the parameter values
param_vals = np.linspace(min_val, max_val, n)

# Generate grid for 3 free parameters using meshgrid
grid = np.array(np.meshgrid(param_vals, param_vals, param_vals, indexing='ij'))
grid = grid.reshape(3, -1).T  # shape: (n^3, 3)

# Append the fixed parameter (rotation = 0, layer = 1)
fixed_params = np.zeros((grid.shape[0], 2))
grid = np.hstack((grid, fixed_params))  # now shape: (n^3, 5)
print(grid.shape)

combos = list(itertools.product(grid, repeat=2))  # List of tuples ((img1_params), (img2_params))

# Convert to NumPy array and reshape to (n^3 * n^3, 2, 5)
stacked_params = np.array(combos)  # shape: (n^6, 2, 5)
print("Stacked parameters shape:", stacked_params.shape)

grid_img1 = stacked_params[:, 0, :]  # shape: (n^6, 5)
grid_img2 = stacked_params[:, 1, :]  # shape: (n^6, 5)


(216, 5)
Stacked parameters shape: (46656, 2, 5)


In [36]:
BATCH = 1024
GAMMA = 0.05

# Set to True to run
RUN = True

if RUN:
    def compute(full_loss=False):
        dir_name = 'loss_visualizer_full_loss' if full_loss else 'loss_visualizer'
        rasterized_imgs = []
        all_empty_pixels = []
        for it, position in enumerate(stacked_params):
            print(f"Computing rasterization #{it}")
            # Reshape (one for each primitive)
            position = position.reshape(-1, 5)

            #position = clip_sol(position, max_val)
            

            img = rasterize_shapes(primitives_selected, position, (HEIGHT, WIDTH))
            np_img = np.array(img)
            all_empty_pixels.append(np.sum(np_img[:, :, 3] == 0) / (np_img.shape[0] * np_img.shape[1])) # (300, 300, 4) -> 300 * 300 = 90000

            rasterized_imgs.append(Image.alpha_composite(
                                        #Image.new("RGBA", img.size, (54, 108, 176, 255)),  # Blue background
                                        Image.new("RGBA", img.size, (255, 255, 255, 255)),  # White background
                                        img
                                    ).convert("RGB"))

            if (it + 1) % BATCH == 0:
                print(f"Computing the {it+1} loss")
                loss_matrix = compute_loss(prompt_embedding=prompt_embedding, rasterized_imgs=rasterized_imgs)
                if full_loss:
                    empty_pixel_loss = torch.tensor(all_empty_pixels).to(DEVICE) # (population_size)
                    loss_matrix = loss_matrix * (1 - GAMMA) + (empty_pixel_loss) * GAMMA
                torch.save(loss_matrix, path.join(RESULTS_DIR, dir_name, f'tensor_{(it + 1) // BATCH}.pt'))
                rasterized_imgs = []
                all_empty_pixels = []

        if len(rasterized_imgs) > 0:
            print(f"Computing the last loss")
            loss_matrix = compute_loss(prompt_embedding=prompt_embedding, rasterized_imgs=rasterized_imgs)
            torch.save(loss_matrix, path.join(RESULTS_DIR, dir_name, 'tensor_last.pt'))

    #compute()
    compute(full_loss=True)


Computing rasterization #0
Computing rasterization #1
Computing rasterization #2
Computing rasterization #3
Computing rasterization #4
Computing rasterization #5
Computing rasterization #6
Computing rasterization #7
Computing rasterization #8
Computing rasterization #9
Computing rasterization #10
Computing rasterization #11
Computing rasterization #12
Computing rasterization #13
Computing rasterization #14
Computing rasterization #15
Computing rasterization #16
Computing rasterization #17
Computing rasterization #18
Computing rasterization #19
Computing rasterization #20
Computing rasterization #21
Computing rasterization #22
Computing rasterization #23
Computing rasterization #24
Computing rasterization #25
Computing rasterization #26
Computing rasterization #27
Computing rasterization #28
Computing rasterization #29
Computing rasterization #30
Computing rasterization #31
Computing rasterization #32
Computing rasterization #33
Computing rasterization #34
Computing rasterization #35
Co

In [30]:
def plot_contour(stacked_loss, title="Loss Landscape Contour Plot"):
    # Create axes for plotting (using flattened indices)
    n_combos = grid_img1.shape[0]
    print(f"Combos: {n_combos}")

    loss_matrix = stacked_loss.reshape(grid.shape[0], -1)

    x_axis = np.arange(n_combos)
    y_axis = np.arange(n_combos)
    fig = go.Figure(data=go.Contour(
        z=loss_matrix.T,
        x=x_axis,
        y=y_axis,
        contours_coloring='heatmap',  # or 'lines', 'none'
        colorbar=dict(title='Loss'),
        line_smoothing=0.85  # optional, for smoother contours
    ))

    # These are the best figures
    k = 1
    values, indices = torch.topk(loss_matrix.view(-1), k)  # Flatten the matrix and get the top k values and their indices

    # Convert flattened indices to row-column indices (if needed)
    row_indices = indices // loss_matrix.size(1)  # Divide by number of columns to get row index
    col_indices = indices % loss_matrix.size(1) 
    hover_labels = [f"x: {x}<br>y: {y}<br>z: {val:.2f}" for val, x, y in zip(values, row_indices, col_indices)]


    fig.add_trace(go.Scatter(
        x=row_indices,
        y=col_indices,
        mode='markers',
        marker=dict(
            symbol='star',
            size=10,
            color='red',
            line=dict(color='black', width=1)
        ),
        hovertext=hover_labels,
        hoverinfo='text',
        showlegend=False
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Primitive#1",
        yaxis_title="Primitive#2",
    )

    fig.show()

    # Compute the flatness metric: Coefficient of Variation (CV)
    # Convert loss_matrix to NumPy array if it's a torch.Tensor
    loss_values = loss_matrix.detach().cpu().numpy() if torch.is_tensor(loss_matrix) else np.array(loss_matrix)
    flat_loss = loss_values.flatten()

    # Compute the standard deviation and mean
    loss_std = np.std(flat_loss)
    loss_mean = np.mean(flat_loss)

    # Handle potential division by zero if loss_mean is nearly zero
    if np.isclose(loss_mean, 0):
        flatness_cv = float('inf')  # or choose an alternative metric
    else:
        flatness_cv = loss_std / loss_mean

    print(f"Flatness Coefficient of Variation (CV): {flatness_cv:.4f}")
        

In [ ]:
def plot_rasterizations(stacked_loss, title="Cosine Similarity"):
    # Sliders for 4 coordinates
    loss_matrix = stacked_loss.reshape(grid.shape[0], -1)
    max_val = loss_matrix.shape[0]-1
    x1 = widgets.IntText(min=0, max=max_val, value=155, description='x1')
    y1 = widgets.IntText(min=0, max=max_val, value=117, description='y1')
    x2 = widgets.IntText(min=0, max=max_val, value=155, description='x2')
    y2 = widgets.IntText(min=0, max=max_val, value=125, description='y2')
    x3 = widgets.IntText(min=0, max=max_val, value=155, description='x3')
    y3 = widgets.IntText(min=0, max=max_val, value=100, description='y3')
    x4 = widgets.IntText(min=0, max=max_val, value=155, description='x4')
    y4 = widgets.IntText(min=0, max=max_val, value=50, description='y4')

    def update_images(x1, y1, x2, y2, x3, y3, x4, y4):
        coords = [(x1, y1), (x2, y2), (x3, y3), (x4, y4)]
        
        grid_params_reshaped = stacked_params.reshape(grid.shape[0], -1, 2, 5)
        
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        
        for ax, (x, y) in zip(axes, coords):
            entry = grid_params_reshaped[x, y, :, :]
            img = rasterize_shapes(primitives_selected, entry, (HEIGHT, WIDTH))
            img_composite = Image.alpha_composite(
                Image.new("RGBA", img.size, (255, 255, 255, 255)), img
            ).convert("RGB")
            ax.imshow(img_composite)
            ax.set_title(f"Figure#1: {x} - Figure#2: {y}\n{title}: {loss_matrix[x, y]:.4f}")
            ax.axis("off")
        
        plt.tight_layout()
        plt.show()

    controls = widgets.HBox([
        widgets.VBox([x1, y1]),
        widgets.VBox([x2, y2]),
        widgets.VBox([x3, y3]),
        widgets.VBox([x4, y4])
    ])

    out = widgets.interactive_output(update_images, {
        'x1': x1, 'y1': y1,
        'x2': x2, 'y2': y2,
        'x3': x3, 'y3': y3,
        'x4': x4, 'y4': y4
    })

    display(controls, out)

In [31]:
# Directory containing the tensors
loss_dir = path.join(RESULTS_DIR, 'loss_visualizer')

# List all tensor files in the directory
tensor_files = [f for f in os.listdir(loss_dir) if f.endswith('.pt')]

# Load and stack tensors
stacked_loss = torch.cat([torch.load(path.join(loss_dir, f), map_location=DEVICE) for f in sorted(tensor_files)], dim=0).cpu()

print("Stacked loss shape:", stacked_loss.shape)
plot_contour(stacked_loss)

Stacked loss shape: torch.Size([46656])
Combos: 46656


Flatness Coefficient of Variation (CV): 0.0583


In [ ]:
# Directory containing the tensors
loss_dir = path.join(RESULTS_DIR, 'loss_visualizer_full_loss')

# List all tensor files in the directory
tensor_files = [f for f in os.listdir(loss_dir) if f.endswith('.pt')]

# Load and stack tensors
stacked_loss = torch.cat([torch.load(path.join(loss_dir, f), map_location=DEVICE) for f in sorted(tensor_files)], dim=0).cpu()

print("Stacked loss shape:", stacked_loss.shape)
plot_contour(stacked_loss, title="Loss Landscape Contour Plot (Full Loss)")
plot_rasterizations(stacked_loss, title="Cosine Similarity (Full Loss)")

Stacked loss shape: torch.Size([46656])
Combos: 46656


Flatness Coefficient of Variation (CV): 0.0444


Output()